# 02 · Causal structure and its validation

The SCM is the audit that binds. A baseline can flip the box on paper by moving a
feature the world would move *back* the moment the applicant acts; only a plan
that agrees with the causal structure actually comes true. This notebook shows
the graphs and how far the data supports them.

The graphs are hand-built from domain knowledge (§6), then **checked** against the
data with conditional-independence tests — not fitted, because a graph fitted to
the same data the black box saw would launder the model's own correlations back
in as "causes".

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
ROOT = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULTS = ROOT / "results"

def show_image(path):
    """Display a saved figure inline in a notebook; no-op-print in a script."""
    path = pathlib.Path(path)
    try:
        from IPython.display import Image, display  # noqa: PLC0415
        display(Image(filename=str(path)))
    except Exception:
        print("figure saved at", path)

def read_result(name):
    """Read a results CSV, or say which experiment writes it."""
    p = RESULTS / f"{name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing - run the experiment that writes it")
    return pd.read_csv(p)

In [ ]:
from src.config import DATASETS
from src.scm import get_scm

rows = []
for d in DATASETS:
    s = get_scm(d).summary()
    rows.append({"dataset": d, "nodes": s["n_nodes"], "edges": s["n_edges"],
                 "features": s["n_features"], "actionable": s["n_actionable"],
                 "acyclic": s["acyclic"]})
pd.DataFrame(rows).set_index("dataset")

In [ ]:
# The primary graph, drawn and saved through the SCM plotting layer.
from src.viz.plot_graphs import save_scm_figure

paths = save_scm_figure("adult")
show_image(paths[0])

## How well does the data back each edge?

Each hypothesised edge implies a conditional (in)dependence the data can be asked
about. The summary below counts how many held. The counts are *evidence about the
graph*, not a score to maximise — a contradicted edge we keep on domain grounds is
a documented modelling choice, not a bug.

In [ ]:
try:
    display(read_result("scm_validation_summary"))
except FileNotFoundError as e:
    print(e)

In [ ]:
# A few individual CI tests, most confident first.
try:
    ci = read_result("scm_ci_tests")
    cols = [c for c in ["dataset", "x", "y", "given", "verdict", "p_value",
                        "statistic"] if c in ci.columns]
    display(ci[cols].head(12))
except FileNotFoundError as e:
    print(e)

## Deviation — graph provenance and limits

- The **Adult** SCM is a 10-node / 25-edge superset of the 8-feature graph
  sketched in §6.1; the extra structure (and the latent node) is documented in the
  README.
- The **German** graph is the weakest: of 37 implied independences, roughly
  **18 are supported, 1 is weak, and 18 are contradicted** by CI tests. It is kept
  as written for domain reasons and flagged as a limitation — the robustness
  experiment (§04) re-runs the whole comparison under a perturbed graph to show the
  conclusions do not hinge on any single contested edge.